# Fine-tuning IndicTrans2 for Sanskrit → English Translation

**Model:** `ai4bharat/indictrans2-indic-en-1B`  
A 1-billion parameter encoder-decoder translation model trained by AI4Bharat on 22 Indian languages including Sanskrit.

**Task:** Fine-tune on Bhagavata Purana verse data:
- **Source:** Sanskrit verse text (Devanagari script)
- **Target:** English translation

**Fine-tuning method:** LoRA (Low-Rank Adaptation) — keeps the 1B base model frozen and trains only ~5M extra parameters, which fits comfortably on a free T4 GPU.

**Evaluation metric:** BLEU score (standard for translation quality). Higher = better.

---
**Before running:** Set runtime to GPU → Runtime → Change runtime type → **T4 GPU**

## Step 1 — Install dependencies

In [ ]:
# Core ML libraries
!pip install transformers==4.40.0 peft accelerate datasets sacrebleu sentencepiece -q

# IndicTransToolkit: official preprocessing for IndicTrans2
!pip install git+https://github.com/VarunGumma/IndicTransToolkit.git -q

print("All packages installed.")

## Step 2 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  : {gpu_name}")
    print(f"VRAM : {vram_gb:.1f} GB")
else:
    print("NO GPU FOUND — go to Runtime → Change runtime type → T4 GPU")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32

## Step 3 — Upload the JSON dataset

1. On your PC, zip the entire `json/` folder → right-click → Send to Compressed file → name it `json_files.zip`
2. Run the cell below, click **Choose Files**, and select `json_files.zip`
3. Wait for upload + extraction to finish

In [ ]:
from google.colab import files
import zipfile, os, glob

print("Upload json_files.zip ...")
uploaded = files.upload()

os.makedirs('json_files', exist_ok=True)
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('json_files/')
        print(f"Extracted: {fname}")

json_paths = glob.glob('json_files/**/*.json', recursive=True) + glob.glob('json_files/*.json')
json_paths = [p for p in json_paths if 'bhagavata_book' in os.path.basename(p)]
print(f"Found {len(json_paths)} chapter files")

## Step 4 — Load and clean the verse-translation pairs

Each JSON entry has:
- `devanagari`: Sanskrit verse in Devanagari script (our **source**)
- `translation`: English translation (our **target**)

We clean the source text by removing embedded verse numbers (॥ १ ॥ etc.).

In [ ]:
import json, re, random
from collections import Counter


# Regex to strip verse number markers like ॥ १ ॥ or ॥ 1 ॥
_VERSE_NUM = re.compile(r'॥[\s\d०-९]+॥')

def clean_devanagari(text: str) -> str:
    """Remove verse number stamps and collapse whitespace."""
    text = _VERSE_NUM.sub('', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


examples = []
skipped  = 0

for path in sorted(json_paths):
    try:
        chapter_data = json.loads(open(path, encoding='utf-8').read())
    except Exception:
        skipped += 1
        continue

    for verse in chapter_data:
        src = clean_devanagari(verse.get('devanagari', '') or '')
        tgt = (verse.get('translation') or '').strip()

        # Skip if either side is empty or very short
        if len(src) < 10 or len(tgt) < 10:
            continue

        examples.append({'source': src, 'target': tgt})

print(f"Total parallel pairs   : {len(examples):,}")
print(f"Files skipped (errors) : {skipped}")

# Show a sample
print("\n--- Sample pair ---")
ex = examples[10]
print(f"Sanskrit : {ex['source'][:120]}...")
print(f"English  : {ex['target'][:120]}...")

## Step 5 — Train / validation split

In [ ]:
random.seed(42)
random.shuffle(examples)

# Use a small fixed validation set so BLEU evaluation is fast
val_size        = min(300, int(0.1 * len(examples)))
val_examples    = examples[:val_size]
train_examples  = examples[val_size:]

print(f"Train : {len(train_examples):,} verse pairs")
print(f"Val   : {len(val_examples):,} verse pairs")

## Step 6 — Load IndicTrans2 model and tokenizer

We load the **Indic→English** direction of IndicTrans2.  
Source language code: `san_Deva` (Sanskrit in Devanagari)  
Target language code: `eng_Latn` (English)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_ID  = "ai4bharat/indictrans2-indic-en-1B"
SRC_LANG  = "san_Deva"
TGT_LANG  = "eng_Latn"
MAX_SRC   = 256
MAX_TGT   = 256

print(f"Loading tokenizer from {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print(f"Loading model (1B params — may take 2-3 minutes) ...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded — {total_params:.0f}M parameters")

## Step 7 — Load IndicProcessor for preprocessing

IndicTransToolkit's `IndicProcessor` normalizes Devanagari text and adds the direction tag that IndicTrans2 expects.

In [ ]:
from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

# Quick sanity check
sample_src = [examples[0]['source']]
processed  = ip.preprocess_batch(sample_src, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
print("Raw input    :", sample_src[0][:80])
print("Preprocessed :", processed[0][:80])

## Step 8 — Baseline BLEU (before fine-tuning)

We run inference on the **validation set** now, before any fine-tuning, to get a baseline BLEU score.  
This lets us see how much the fine-tuning actually improves the model on our specific domain.

In [ ]:
import sacrebleu
from tqdm.auto import tqdm


def translate_batch(source_texts, batch_size=8):
    """Translate a list of Sanskrit strings to English using the current model state."""
    all_translations = []
    model.eval()

    for i in tqdm(range(0, len(source_texts), batch_size), desc="Translating"):
        batch_src = source_texts[i : i + batch_size]

        # Preprocess
        processed = ip.preprocess_batch(batch_src, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

        # Tokenize
        inputs = tokenizer(
            processed,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SRC,
        ).to(DEVICE)

        # Generate
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                num_beams=4,
                max_length=MAX_TGT,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            )

        # Decode
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        decoded = ip.postprocess_batch(decoded, lang=TGT_LANG)
        all_translations.extend(decoded)

    return all_translations


val_sources    = [e['source'] for e in val_examples]
val_references = [e['target'] for e in val_examples]

print("Running baseline inference on validation set ...")
baseline_translations = translate_batch(val_sources, batch_size=4)

# BLEU score
baseline_bleu = sacrebleu.corpus_bleu(
    baseline_translations,
    [val_references],
    tokenize='13a',
)
print(f"\nBASELINE BLEU (before fine-tuning): {baseline_bleu.score:.2f}")

## Step 9 — Show a baseline translation sample

In [ ]:
print("=" * 70)
print("SAMPLE TRANSLATIONS (baseline — before fine-tuning)")
print("=" * 70)
for i in range(5):
    print(f"\n[{i+1}] Sanskrit  : {val_sources[i][:100]}")
    print(f"    Reference : {val_references[i][:100]}")
    print(f"    Model out : {baseline_translations[i][:100]}")

## Step 10 — Apply LoRA for efficient fine-tuning

LoRA adds small trainable matrices (rank=16) to the attention layers. The 1B base weights stay **frozen** — only the LoRA weights (~10M parameters) are trained. This fits on a T4 GPU in under 10 minutes.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Find the correct attention module names for this model
print("Attention-related modules in the model:")
attn_modules = [
    name for name, _ in model.named_modules()
    if any(k in name for k in ['q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2'])
    and '.' not in name.split('.')[-1]  # only leaf names
]
# Get the unique leaf module names
target_leaf_names = list(dict.fromkeys(
    name.split('.')[-1] for name in attn_modules
))
print(f"  Unique names: {target_leaf_names}")

# Use standard NLLB-style attention names; fall back to whatever we found
TARGET_MODULES = [n for n in ['q_proj', 'k_proj', 'v_proj', 'out_proj'] if n in target_leaf_names]
if not TARGET_MODULES:
    TARGET_MODULES = target_leaf_names[:4]  # fallback: first 4 found
print(f"  → Using: {TARGET_MODULES}")

lora_config = LoraConfig(
    task_type   = TaskType.SEQ_2_SEQ_LM,
    r           = 16,          # LoRA rank — higher = more capacity, more memory
    lora_alpha  = 32,          # scaling factor
    lora_dropout= 0.05,
    bias        = "none",
    target_modules = TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 11 — Build the training dataset

We preprocess source texts with `IndicProcessor` then tokenize both source and target with the IndicTrans2 tokenizer. Labels use `-100` for padding positions so the loss function ignores them.

In [ ]:
from datasets import Dataset


def preprocess_and_tokenize(examples_batch):
    src_texts = examples_batch['source']
    tgt_texts = examples_batch['target']

    # IndicProcessor normalization on source side
    processed_src = ip.preprocess_batch(src_texts, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

    # Tokenize source
    model_inputs = tokenizer(
        processed_src,
        max_length=MAX_SRC,
        truncation=True,
        padding='max_length',
    )

    # Tokenize target
    target_enc = tokenizer(
        text_target=tgt_texts,
        max_length=MAX_TGT,
        truncation=True,
        padding='max_length',
    )

    # Replace padding token id in labels with -100 (ignored by CrossEntropyLoss)
    labels = target_enc['input_ids']
    labels = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels
    ]
    model_inputs['labels'] = labels
    return model_inputs


def make_dataset(example_list):
    raw = Dataset.from_list(example_list)
    return raw.map(
        preprocess_and_tokenize,
        batched=True,
        batch_size=64,
        remove_columns=['source', 'target'],
        desc="Tokenizing",
    )


print("Building train dataset ...")
train_ds = make_dataset(train_examples)
print("Building val dataset ...")
val_ds   = make_dataset(val_examples)

print(f"\nTrain : {len(train_ds):,} examples")
print(f"Val   : {len(val_ds):,} examples")
print(f"Columns: {train_ds.column_names}")

## Step 12 — Define BLEU metric for the Trainer

In [ ]:
import numpy as np


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels (padding) then decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process with IndicProcessor
    decoded_preds = ip.postprocess_batch(decoded_preds, lang=TGT_LANG)

    # BLEU — n-gram overlap score (0–100)
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels], tokenize='13a')

    # ChrF — character-level F-score (0–100, more forgiving than BLEU)
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels])

    return {
        'bleu': round(bleu.score, 4),
        'chrf': round(chrf.score, 4),
    }

## Step 13 — Fine-tune with Seq2SeqTrainer

Training for **3 epochs** on a T4 GPU takes approximately **10–25 minutes** depending on dataset size.

Key settings:
- `predict_with_generate=True` — the Trainer generates real translations for BLEU computation (not teacher-forced logits)
- `generation_max_length=256` — max output length during eval generation
- `fp16=True` — half precision for speed and memory

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = './indictrans2-sanskrit-en',
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 4,     # effective batch = 4 × 4 = 16
    learning_rate               = 5e-5,
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,
    fp16                        = (DEVICE == 'cuda'),
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    predict_with_generate       = True,
    generation_max_length       = MAX_TGT,
    logging_steps               = 50,
    report_to                   = 'none',
    save_total_limit            = 2,
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

print("Starting fine-tuning ...")
trainer.train()

## Step 14 — Final evaluation: BLEU before vs after

In [ ]:
print("Running final evaluation on validation set ...")
final_results = trainer.evaluate()

baseline_chrf = sacrebleu.corpus_chrf(baseline_translations, [val_references])

print()
print("=" * 55)
print("RESULTS SUMMARY")
print("=" * 55)
print(f"  {'Metric':<28} {'Before':>8}  {'After':>8}")
print("-" * 55)
print(f"  {'BLEU (0–100)':<28} {baseline_bleu.score:>8.2f}  {final_results['eval_bleu']:>8.2f}")
print(f"  {'ChrF (0–100)':<28} {baseline_chrf.score:>8.2f}  {final_results['eval_chrf']:>8.2f}")
delta_bleu = final_results['eval_bleu'] - baseline_bleu.score
delta_chrf = final_results['eval_chrf'] - baseline_chrf.score
print("-" * 55)
print(f"  {'BLEU improvement':<28} {delta_bleu:>+17.2f}")
print(f"  {'ChrF improvement':<28} {delta_chrf:>+17.2f}")
print("=" * 55)

## Step 14b — Full metrics dashboard (all accuracy-style numbers explained)

This cell computes **four different ways to measure translation quality**, each showing a different aspect of accuracy.

In [ ]:
# Generate fine-tuned translations for the full validation set
print("Generating fine-tuned translations for full metrics ...")
ft_translations_full = translate_batch(val_sources, batch_size=4)

# ── 1. BLEU (already computed above) ──────────────────────────────────────────
ft_bleu   = sacrebleu.corpus_bleu(ft_translations_full, [val_references], tokenize='13a')

# ── 2. ChrF ── character-level F-score ────────────────────────────────────────
# Measures character n-gram overlap. Less sensitive to exact word order.
# Ranges 0–100. Easier to get a higher score than BLEU.
ft_chrf   = sacrebleu.corpus_chrf(ft_translations_full, [val_references])
bl_chrf   = sacrebleu.corpus_chrf(baseline_translations, [val_references])

# ── 3. Word Precision / Recall / F1 ───────────────────────────────────────────
# For each sentence: what fraction of predicted words appear in the reference (precision)?
# And what fraction of reference words were predicted (recall)?
def word_pr_f1(hypotheses, references):
    precisions, recalls, f1s = [], [], []
    for hyp, ref in zip(hypotheses, references):
        hyp_words = set(hyp.lower().split())
        ref_words = set(ref.lower().split())
        if not hyp_words:
            precisions.append(0); recalls.append(0); f1s.append(0)
            continue
        tp = len(hyp_words & ref_words)
        p  = tp / len(hyp_words)
        r  = tp / len(ref_words) if ref_words else 0
        f  = 2 * p * r / (p + r) if (p + r) > 0 else 0
        precisions.append(p); recalls.append(r); f1s.append(f)
    return (
        sum(precisions) / len(precisions),
        sum(recalls)    / len(recalls),
        sum(f1s)        / len(f1s),
    )

bl_prec, bl_rec, bl_f1 = word_pr_f1(baseline_translations,  val_references)
ft_prec, ft_rec, ft_f1 = word_pr_f1(ft_translations_full,   val_references)

# ── 4. "Good translation" rate ─────────────────────────────────────────────────
# % of sentences with sentence-BLEU >= 20  (considered a usable translation)
def good_rate(hypotheses, references, threshold=20):
    good = 0
    for hyp, ref in zip(hypotheses, references):
        s = sacrebleu.sentence_bleu(hyp, [ref], smooth_method='exp').score
        if s >= threshold:
            good += 1
    return good / len(hypotheses) * 100

bl_good = good_rate(baseline_translations, val_references)
ft_good = good_rate(ft_translations_full,  val_references)

# ── Print the full dashboard ───────────────────────────────────────────────────
print()
print("╔" + "═"*65 + "╗")
print("║  FULL METRICS DASHBOARD — IndicTrans2 Sanskrit→English        ║")
print("╠" + "═"*65 + "╣")
print(f"║  {'Metric':<35} {'Baseline':>10}  {'Fine-tuned':>10}  ║")
print("╠" + "═"*65 + "╣")
print(f"║  {'BLEU  (0–100, higher=better)':<35} {baseline_bleu.score:>10.2f}  {ft_bleu.score:>10.2f}  ║")
print(f"║  {'ChrF  (0–100, higher=better)':<35} {bl_chrf.score:>10.2f}  {ft_chrf.score:>10.2f}  ║")
print(f"║  {'Word Precision  (%)':<35} {bl_prec*100:>10.1f}  {ft_prec*100:>10.1f}  ║")
print(f"║  {'Word Recall     (%)':<35} {bl_rec*100:>10.1f}  {ft_rec*100:>10.1f}  ║")
print(f"║  {'Word F1         (%)':<35} {bl_f1*100:>10.1f}  {ft_f1*100:>10.1f}  ║")
print(f"║  {'Good translations (BLEU≥20, %)':<35} {bl_good:>10.1f}  {ft_good:>10.1f}  ║")
print("╚" + "═"*65 + "╝")

print("""
What each metric means:
  BLEU            — Standard MT metric. Counts n-gram word sequences that
                    match the reference. A gain of +5 points is significant.
  ChrF            — Same idea but at character level. More lenient (Sanskrit
                    proper nouns are often partially right). Easier to read.
  Word Precision  — Of all words the model produced, what % were in the ref?
                    High precision = model doesn't hallucinate extra words.
  Word Recall     — Of all words in the reference, what % did the model say?
                    High recall = model captures the meaning.
  Word F1         — Harmonic mean of Precision & Recall. Best single number
                    for "how many correct words" — read this like accuracy %.
  Good trans. %   — % of verses where the model scored BLEU≥20 (usable).
                    This is the closest thing to a pass/fail accuracy.
""")

## Step 15 — Sample translations: before vs after

In [ ]:
print("Generating post-fine-tuning translations on the same 5 samples ...")
fine_tuned_translations = translate_batch(val_sources[:5], batch_size=4)

print()
print("=" * 70)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)

for i in range(5):
    print(f"\n[{i+1}]")
    print(f"  Sanskrit  : {val_sources[i][:110]}")
    print(f"  Reference : {val_references[i][:110]}")
    print(f"  Baseline  : {baseline_translations[i][:110]}")
    print(f"  Fine-tuned: {fine_tuned_translations[i][:110]}")

## Step 16 — Sentence-level BLEU distribution

In [ ]:
import matplotlib.pyplot as plt

# Compute sentence-level BLEU for every validation example
print("Computing sentence-level BLEU on full validation set ...")
ft_translations = translate_batch(val_sources, batch_size=4)

sentence_bleus_baseline = []
sentence_bleus_ft       = []

for hyp_base, hyp_ft, ref in zip(baseline_translations, ft_translations, val_references):
    b_base = sacrebleu.sentence_bleu(hyp_base, [ref], smooth_method='exp').score
    b_ft   = sacrebleu.sentence_bleu(hyp_ft,   [ref], smooth_method='exp').score
    sentence_bleus_baseline.append(b_base)
    sentence_bleus_ft.append(b_ft)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, scores, title, color in [
    (axes[0], sentence_bleus_baseline, f'Baseline BLEU\n(mean={sum(sentence_bleus_baseline)/len(sentence_bleus_baseline):.1f})', '#d62728'),
    (axes[1], sentence_bleus_ft,       f'Fine-tuned BLEU\n(mean={sum(sentence_bleus_ft)/len(sentence_bleus_ft):.1f})',         '#1f77b4'),
]:
    ax.hist(scores, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Sentence BLEU', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(title, fontsize=12)

plt.suptitle('IndicTrans2 — Sanskrit→English BLEU Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('bleu_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved as bleu_distribution.png")

## Step 17 — Translate a custom Sanskrit verse

In [ ]:
def translate_one(sanskrit_devanagari: str) -> str:
    """Translate a single Sanskrit verse (Devanagari) to English."""
    result = translate_batch([sanskrit_devanagari], batch_size=1)
    return result[0]


# Try with a verse from the dataset (paste any Devanagari text here)
my_verse = examples[50]['source']
my_ref   = examples[50]['target']

print("Custom translation demo")
print("-" * 60)
print(f"Input     : {my_verse}")
print(f"Reference : {my_ref}")
print(f"Output    : {translate_one(my_verse)}")

## Step 18 — Save and download the fine-tuned model

In [ ]:
import shutil

SAVE_DIR = 'indictrans2-sanskrit-en-lora'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save results summary
summary = {
    'model'              : MODEL_ID,
    'src_lang'           : SRC_LANG,
    'tgt_lang'           : TGT_LANG,
    'train_examples'     : len(train_examples),
    'val_examples'       : len(val_examples),
    'baseline_bleu'      : round(baseline_bleu.score, 4),
    'fine_tuned_bleu'    : round(final_results['eval_bleu'], 4),
    'bleu_improvement'   : round(final_results['eval_bleu'] - baseline_bleu.score, 4),
}
with open(f'{SAVE_DIR}/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Results summary:")
print(json.dumps(summary, indent=2))

# Zip and download
shutil.make_archive('indictrans2_sanskrit_lora', 'zip', SAVE_DIR)
files.download('indictrans2_sanskrit_lora.zip')
print("\nDownload started.")